In [ ]:
%%capture
!pip install timm
!pip install --upgrade ultralytics

In [2]:
import cv2
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import numpy as np
from PIL import Image
import json
import os
import torch.nn.functional as F
import timm
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\ACER\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
class JsonPrototypePipeline:
    def __init__(self,  yolo_model, extractor, transform, threshold=0.5):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.threshold = threshold
        print(f"Initializing Pipeline on {self.device}...")

        # 1. Load Models
        self.detector = yolo_model
        self.extractor = extractor.to(self.device)
        self.extractor.eval()

        # 2. Preprocessing
        self.transform = transform

        # 3. Reference Storage
        # Store as a tensor matrix for vectorized comparison [N_refs, Emb_Dim]
        self.ref_tensor = None

    def assign_ref_img(self, ref_image_paths):
        self.process_reference_images(ref_image_paths)

    def get_embedding_batch(self, img_tensors):
        """
        Runs the extractor on a batch of images.
        Input: Tensor of shape [Batch_Size, C, H, W]
        Output: Tensor of shape [Batch_Size, Embed_Dim] (Normalized)
        """
        with torch.inference_mode():
            features = self.extractor(img_tensors)
            # Flatten if necessary (depending on model architecture)
            if len(features.shape) > 2:
                features = features.flatten(start_dim=1)
            # Normalize features for Cosine Similarity
            features = F.normalize(features, p=2, dim=1)
            return features

    def process_reference_images(self, paths):
        ref_embeddings = []
        for i, path in enumerate(paths):
            try:
                img = Image.open(path).convert('RGB')
                # Transform and add batch dimension
                img_t = self.transform(img).unsqueeze(0).to(self.device)

                # Extract and normalize
                with torch.inference_mode():
                    emb = self.extractor(img_t).flatten(start_dim=1)
                    emb = F.normalize(emb, p=2, dim=1)

                ref_embeddings.append(emb)
            except Exception as e:
                print(f"Error loading {path}: {e}")

        if ref_embeddings:
            # Stack into a single matrix: [N_refs, Embed_Dim]
            self.ref_tensor = torch.cat(ref_embeddings, dim=0)

    def process_batch(self, video_paths, video_name, output_json='/content/predictions.json'):
        final_output = []

        # Ensure references exist
        if self.ref_tensor is None:
            print("No reference images assigned. Skipping.")
            return

        for video_path in video_paths:
            # Initialize storage for this video
            object_tracks = []

            cap = cv2.VideoCapture(video_path)
            frame_idx = 0

            while cap.isOpened():
                ret, frame = cap.read()
                if not ret:
                    break
                h, w, _ = frame.shape

                # YOLO Inference
                results = self.detector(frame, verbose=False, conf=0.1)

                # Collect crops for batch processing
                crops = []
                coords = []

                for result in results:
                    if result.boxes is None: continue

                    # Get boxes in one go (on CPU for slicing)
                    boxes = result.boxes.xyxy.cpu().numpy().astype(int)

                    for box in boxes:
                        x1, y1, x2, y2 = box

                        # Clamp coordinates
                        x1, y1 = max(0, x1), max(0, y1)
                        x2, y2 = min(w, x2), min(h, y2)

                        # Basic size check
                        if x2 - x1 < 2 or y2 - y1 < 2:
                            continue

                        # Crop (Keep numpy for now)
                        obj_crop = frame[y1:y2, x1:x2]

                        # Convert to PIL for Transform (to match original logic)
                        # Note: This part is CPU bound, difficult to optimize without changing transform logic completely
                        obj_crop_pil = Image.fromarray(cv2.cvtColor(obj_crop, cv2.COLOR_BGR2RGB))

                        crops.append(self.transform(obj_crop_pil))
                        coords.append((x1, y1, x2, y2))

                # Batch Inference
                if crops:
                    batch_t = torch.stack(crops).to(self.device)

                    batch_emb = self.get_embedding_batch(batch_t)

                    sim_matrix = torch.mm(batch_emb, self.ref_tensor.T)

                    max_vals, max_indices = torch.max(sim_matrix, dim=1)
                    max_vals_cpu = max_vals.cpu().numpy()


                    for idx, score in enumerate(max_vals_cpu):

                        # if score > self.threshold: <--- khỏi filter thì tốt hơn
                            x1, y1, x2, y2 = coords[idx]
                            bbox_entry = {
                                "frame": frame_idx,
                                "x1": int(x1), "y1": int(y1),
                                "x2": int(x2), "y2": int(y2)
                            }
                            object_tracks.append(bbox_entry)

                frame_idx += 1
            cap.release()


            video_detections_dict = {}

            for bboxes in object_tracks:
                if bboxes:
                    if "bboxes" not in video_detections_dict:
                        video_detections_dict["bboxes"] = []
                    video_detections_dict["bboxes"].append(bboxes)

            new_entry = {
                "video_id": video_name,
                "detections": [video_detections_dict]
            }


            current_data = []
            if os.path.exists(output_json):
                try:
                    with open(output_json, 'r') as f:
                        current_data = json.load(f)
                except json.JSONDecodeError:
                    current_data = []

            current_data.append(new_entry)

            with open(output_json, 'w') as f:
                json.dump(current_data, f, indent=4)

            print(f"Appended {video_name} to {output_json}")

In [4]:
class FPNWrapper(nn.Module):
    def __init__(self, backbone, out_channels=256):
        super().__init__()
        # Assume backbone returns feature maps from 3 stages
        self.backbone = backbone
        self.lateral3 = nn.Conv2d(48, out_channels, 1)
        self.lateral4 = nn.Conv2d(80, out_channels, 1)
        self.lateral5 = nn.Conv2d(960, out_channels, 1)
        self.smoothing3 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.smoothing4 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.smoothing5 = nn.Conv2d(out_channels, out_channels, 3, padding=1)

    def forward(self, x):
        # Example: features from conv3, conv4, conv5
        c3, c4, c5 = self.backbone(x)
        p5 = self.lateral5(c5)
        p5 = self.smoothing5(p5)
        p4 = self.lateral4(c4) + F.interpolate(p5, size=c4.shape[-2:], mode='nearest')
        p4 = self.smoothing4(p4)
        p3 = self.lateral3(c3) + F.interpolate(p4, size=c3.shape[-2:], mode='nearest')
        p3 = self.smoothing3(p3)
        return [p3, p4, p5]

class AttentionPooling(nn.Module):
    def __init__(self, in_channels=256):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // 8, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // 8, 1, 1),
            nn.Sigmoid()
        )

    def forward(self, feat):
        attn_map = self.attn(feat)
        weighted = (feat * attn_map).sum(dim=(2, 3)) / (attn_map.sum(dim=(2, 3)) + 1e-6)
        return weighted, attn_map

class PrototypeExtractor(nn.Module):
    def __init__(self, fpn, attn_pool, output_size=(7, 7)):
        super().__init__()
        self.fpn = fpn
        self.attn_pool = attn_pool
        self.output_size = output_size

    def forward(self, support_patches: torch.Tensor):
        """
        Args:
            support_patches: Tensor of shape [B, C, H, W]
                             where B is the number of support shots.
        Returns:
            proto: Tensor of shape [B, C_out] (The averaged prototype vector)
        """

        # Safety check for empty input
        if support_patches.size(0) == 0:
            return None


        fpn_feats = self.fpn(support_patches)

        roi_feat = fpn_feats[-1]
        if roi_feat.shape[-2:] != self.output_size:
             roi_feat = F.adaptive_avg_pool2d(roi_feat, self.output_size)

        proto, _ = self.attn_pool(roi_feat)


        return proto

class Backbone(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model


    def forward(self, x):
        # Return feature maps from 3 stages
        c3, c4, c5 = self.model(x)
        return c3, c4, c5

    def destroy_hooks(self):
        # Call this method to remove hooks when done
        self.handle1.remove()
        self.handle2.remove()
        self.handle3.remove()

In [ ]:
if __name__ == "__main__":

    ## Mấy cái model trong link drive trên gr
    detection_model_path = "../models/detection_model/best.pt"
    comparing_model_path = "../models/comparing_model/best.pth"
    device = "cuda" if torch.cuda.is_available() else "cpu"

    yolo_model = YOLO(detection_model_path)

    model = timm.create_model(
      "hf_hub:timm/mobilenetv4_conv_medium.e500_r224_in1k",
        pretrained=True,
        features_only=True,
        out_indices=(1, 2, 4)
    ).to(device)
    model.eval()
    backbone = Backbone(model)
    fpn = FPNWrapper(backbone)
    att = AttentionPooling()
    prototypeExtractor = PrototypeExtractor(fpn, att)

    data_config = timm.data.resolve_model_data_config(model)
    transforms = timm.data.create_transform(**data_config, is_training=False)

    if os.path.exists(comparing_model_path):
        print(f"Loading checkpoint from {comparing_model_path}...")
        checkpoint = torch.load(comparing_model_path, map_location=device)

        # A. Load Model Weights
        prototypeExtractor.load_state_dict(checkpoint['model_state_dict'])
        prototypeExtractor.to(device)
    prototypeExtractor.eval()

    pipeline = JsonPrototypePipeline(
        yolo_model = yolo_model,
        extractor = prototypeExtractor,
        transform = transforms,
        threshold=0.70)

    test_data_path = "../data/observing/public_test/samples/"
    ref_images = []
    video_list = []
    for folder in os.listdir(test_data_path):
      object_img_paths = []
      img_fol = test_data_path  + folder + "/" + "object_images/"
      for object_img in os.listdir(img_fol):
        object_img_paths.append(img_fol + object_img)

      video_path = test_data_path  + folder + "/" + "drone_video.mp4"
      print(f"Processing {folder} ...")
      pipeline.assign_ref_img(object_img_paths)
      pipeline.process_batch(video_paths=[video_path], video_name = folder)

      # break


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/39.2M [00:00<?, ?B/s]

Unexpected keys (classifier.bias, classifier.weight, conv_head.weight, norm_head.bias, norm_head.num_batches_tracked, norm_head.running_mean, norm_head.running_var, norm_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


Loading checkpoint from ../models/comparing_model/best.pth...


RuntimeError: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.

In [ ]:
device

'cuda'